# Lab 3.2 — Targeted Behavior
### Path-Specific Rules & Plan Mode Workflows — Reflection & Self-Check Answers

## Exercise 1 — Path-specific rules

**Q1. Why put SECURITY-CRITICAL rules in `src/auth/CLAUDE.md` instead of the root `CLAUDE.md` — what do you gain by scoping them to the path?**

If the strict "never weaken a credential check" rule lived in the root file, it would either apply everywhere (so trivial orders/ helpers get needlessly scrutinized, training everyone to skim past the rule) or get diluted into generic advice so it doesn't feel out of place next to ordinary conventions. Putting it in `src/auth/CLAUDE.md` means the extra caution only loads into context when someone is actually editing token-verification code, and it sits right next to the code it governs, so it is easy to find, review, and keep in sync with the code as auth/ evolves.

**Q2. The auth request was challenged while the orders helper was made cleanly. What made Claude treat them differently, and why is "refuse and offer a safe alternative" the right behaviour?**

Claude Code layers the root `CLAUDE.md` with whichever module `CLAUDE.md` is nearest to the file being edited. `count_items` touched `src/orders/service.py`, so only the general + order-conventions rules were in context, none of which discourage a small typed helper. `verify_token` in `src/auth/tokens.py` loaded `src/auth/CLAUDE.md`, whose explicit rule is "never weaken a token/credential check." Refusing (rather than complying) is correct because the request's literal effect is a real production security regression — silently doing it would let weaker tokens through in prod. Offering a safe alternative (a valid fake token like `npk_test_abcdef123456` for tests) still solves the user's actual underlying need (easier testing) without touching the production check.

**Q3. A strict rule in `src/payments/CLAUDE.md` and a looser root rule could seem to conflict. How does path scoping decide which applies?**

There's no real conflict once you see the layering as additive, not competing: both the root rules and the nearest directory's rules apply at once, and where the directory rule is stricter or more specific, it's the one that actually binds for that path. The root `CLAUDE.md` sets a floor ("every behaviour change ships with a test"); `src/payments/CLAUDE.md` raises the bar for that specific directory ("needs a test covering a successful charge AND a rejected one"). Proximity to the file being edited is the tiebreaker — the closer, more specific file wins whenever it says more or says something stricter than the general rule.

## Exercise 2 — Plan mode for a multi-file migration

**Q1. Why is Plan mode worth the extra step for a multi-file migration, when you could just ask for the edits directly?**

A migration like this touches three files (`auth/tokens.py`, `orders/service.py`, `payments/charges.py`) that aren't all visible in one place, so it's easy to miss a call site or delete the deprecated function before every caller is actually migrated — silently breaking `orders/service.py` or `payments/charges.py`, both of which are money- or security-adjacent. Plan mode surfaces the full call-site list, the intended edits, and the removal step *before* anything changes, so you can catch a missed caller or a wrong approach by reading a few lines instead of by discovering a broken import at runtime.

**Q2. The plan lists "run the tests" as an explicit step. Why bake verification into the plan rather than assume it?**

"Assume it" quietly depends on whoever executes the plan remembering to check, every time — exactly the kind of tacit step that gets skipped under time pressure. Writing it into the plan makes verification a required, visible deliverable of the migration rather than an afterthought, and it gives you a concrete, objective signal (`pytest -q` → `5 passed`) that the refactor didn't change behaviour, instead of trusting that the mechanical import/call swap was done correctly by eye.

**Q3. After migrating all callers, `verify_token_v1` became dead code and was removed. Why is removal part of finishing the migration, not optional cleanup?**

A migration's entire purpose was to get everyone off the weak check; leaving `verify_token_v1` in place after that succeeds keeps a working, importable, weaker credential check sitting in the codebase for the next person (or the next AI-assisted edit) to accidentally call. That's a live security liability, not tidiness. Removing it is what actually closes the gap the migration was meant to close — until it's gone, the deprecated path is still one `import` away from being reintroduced.

## Exercise 3 — Explore before you change

**Q1. The explorer's tools are `Read`, `Grep`, `Glob` — no edit/write/bash. Why constrain a subagent like that?**

The whole point of exploring first is to gather trustworthy context before any change lands under money-critical rules; if the survey step itself could edit files or run arbitrary commands, a misread instruction or an ambiguous prompt during exploration could cause exactly the kind of unreviewed change the lab is trying to prevent. Limiting it to read-only tools makes the guarantee "exploration cannot change anything" enforced by the harness, not by hoping the subagent behaves — the same least-privilege logic as scoping a slash command's `allowed-tools`.

**Q2. Why run exploration in a separate subagent instead of having the main agent read all the files itself?**

Running it as a subagent keeps the main agent's context clean: the subagent can read every file in `src/payments`, chase down `CLAUDE.md`, and grep for deprecated usage, then hand back a short, structured report (files, public API, dependencies, watch-outs) instead of dumping every file's raw contents into the main conversation. It also cleanly separates the read-only survey phase from the edit phase, so the record of "what was explored" versus "what was changed" stays legible.

**Q3. The explorer flagged the money-critical rules and the dependency on `auth` before any edit. How does "explore first" change the quality of the change that follows?**

Going in blind, it would have been easy to add the `$10,000` check without also confirming the module's other invariants — that `charge()` verifies the token first, that money is `Decimal` not `float`, that both the accepted and rejected paths need tests. Because the survey surfaced `src/payments/CLAUDE.md`'s rules and confirmed the existing `verify_token` dependency up front, the follow-up edit could be scoped precisely to what was asked (the upper bound) without disturbing the surrounding invariants, and the test could be written to satisfy the rule ("successful charge AND rejected one") rather than half-covering it.

## 5.1 Self-Check Before You Leave

**1. Where do path-specific rules live, and how does Claude Code decide which `CLAUDE.md` applies to a given file?**

They live in a `CLAUDE.md` placed inside the module they govern — `src/auth/CLAUDE.md`, `src/orders/CLAUDE.md`, `src/payments/CLAUDE.md` — alongside the root `CLAUDE.md` for general rules. Claude Code layers the root file with the nearest directory `CLAUDE.md` to the file being edited, so the rules that load into context are whichever ones are closest to, and therefore most relevant to, the code actually being touched.

**2. Why does the same weakening request get challenged under `auth/` but a helper gets added cleanly under `orders/`?**

Because different rules load for each path: editing `src/auth/tokens.py` pulls in `src/auth/CLAUDE.md`'s explicit "never weaken a token/credential check" rule, while editing `src/orders/service.py` only pulls in the general + order-conventions rules, none of which forbid a small typed helper. Same kind of request, different risk at that path, so different treatment.

**3. What does Plan mode change about how edits happen, and how do you enter it?**

In Plan mode, Claude proposes a full multi-file plan and waits for explicit approval before making any edit, instead of editing as it goes. Enter it with `Shift+Tab` to cycle modes, or start Claude Code directly with `claude --permission-mode plan`.

**4. Why is "run the tests" part of the migration plan, and why remove the deprecated function?**

Running the tests turns "the refactor looks right" into a verified fact (`pytest -q` passing) instead of an assumption, catching a missed call site or behaviour change immediately. Removing `verify_token_v1` is what actually finishes the migration — a working deprecated function left behind after all its callers are gone is dead code that can still be imported and misused, undoing the point of migrating off it.

**5. What is the explorer subagent for, why is it read-only, and why run exploration separately from the main agent?**

It surveys an unfamiliar module (files, public API, dependencies, risks) and reports back without changing anything, so the change that follows is informed by real context instead of guesswork. It's read-only (`Read`, `Grep`, `Glob` only, no edit/write/bash) so the survey step is structurally incapable of making an unreviewed change under strict rules — the guarantee is enforced by tool access, not just instructions. Running it as a separate subagent keeps the main agent's context focused on a concise structured report rather than every file it read, and cleanly separates "look" from "touch."